In [ ]:


#***********************************************************************************************************************************************************


                      #DROUGHT CLASSIFICATION AND PREDICTION USING MACHINE LEARNING


#*************************************************************************************************************************************************************









import pandas as pd

data_dir = r"C:\Users\niran\Downloads\Drought prediction_ML\dataset1"

# Only pull the weather columns that matter most for drought
usecols = ['fips', 'date', 'PRECTOT', 'T2M', 'T2M_MAX', 'T2M_MIN', 'QV2M', 'WS10M', 'score']

train = pd.read_csv(f"{data_dir}/train_timeseries.csv", usecols=usecols)
soil = pd.read_csv(f"{data_dir}/soil_data.csv")

print(train.shape)

(19300680, 9)


In [18]:
train["fips"].nunique()

3108

In [20]:
all_fips = train['fips'].unique()
print(f"Total counties: {len(all_fips)}")

# Take ~150 counties — enough variety to be meaningful, small enough to run fast
subset_fips = all_fips[:150]
train = train[train['fips'].isin(subset_fips)].copy()

print(train.shape)

Total counties: 150
(931500, 9)


In [21]:
train = train.merge(soil, on='fips', how='left')
train['date'] = pd.to_datetime(train['date'])
train = train.sort_values(['fips', 'date']).reset_index(drop=True)

print(train.shape)
print(train.head())

(931500, 40)
   fips       date  PRECTOT   QV2M    T2M  T2M_MAX  T2M_MIN  WS10M  score  \
0  1001 2000-01-01     0.22   9.65  14.74    20.96    11.46   2.20    NaN   
1  1001 2000-01-02     0.20  10.42  16.69    22.80    12.61   2.52    NaN   
2  1001 2000-01-03     3.65  11.76  18.49    22.73    15.32   4.03    NaN   
3  1001 2000-01-04    15.95   6.42  11.40    18.09     2.16   3.84    1.0   
4  1001 2000-01-05     0.00   2.95   3.86    10.82    -2.66   1.60    NaN   

         lat  ...  CULTRF_LAND  CULTIR_LAND  CULT_LAND  SQ1  SQ2  SQ3  SQ4  \
0  32.536382  ...    56.293411     1.014811  57.308224    1    1    1    1   
1  32.536382  ...    56.293411     1.014811  57.308224    1    1    1    1   
2  32.536382  ...    56.293411     1.014811  57.308224    1    1    1    1   
3  32.536382  ...    56.293411     1.014811  57.308224    1    1    1    1   
4  32.536382  ...    56.293411     1.014811  57.308224    1    1    1    1   

   SQ5  SQ6  SQ7  
0    1    1    2  
1    1    1    2 

In [22]:
weather_cols = ['PRECTOT', 'T2M', 'T2M_MAX', 'T2M_MIN', 'QV2M', 'WS10M']

roll_mean = train.groupby('fips')[weather_cols].rolling(window=30, min_periods=1).mean().reset_index(drop=True)
roll_mean.columns = [f"{c}_mean_30d" for c in weather_cols]

train = pd.concat([train, roll_mean], axis=1)
print(train.shape)

(931500, 46)


In [23]:
print(train[['fips', 'date', 'PRECTOT', 'PRECTOT_mean_30d']].head(10))

   fips       date  PRECTOT  PRECTOT_mean_30d
0  1001 2000-01-01     0.22          0.220000
1  1001 2000-01-02     0.20          0.210000
2  1001 2000-01-03     3.65          1.356667
3  1001 2000-01-04    15.95          5.005000
4  1001 2000-01-05     0.00          4.004000
5  1001 2000-01-06     0.01          3.338333
6  1001 2000-01-07     0.01          2.862857
7  1001 2000-01-08     1.02          2.632500
8  1001 2000-01-09    16.19          4.138889
9  1001 2000-01-10    21.23          5.848000


In [24]:
# Keep only rows where we actually have a drought label
labeled = train.dropna(subset=['score']).copy()

# Round score into clean integer classes (0-5)
labeled['score_class'] = labeled['score'].round().astype(int)

print(labeled.shape)
print(labeled['score_class'].value_counts().sort_index())

(133050, 47)
score_class
0    73834
1    23312
2    16112
3    11331
4     5809
5     2652
Name: count, dtype: int64
